In [ ]:
# 1. Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


# 2. Load dataset
data = pd.read_csv("creditcard.csv").dropna()

# 3. Separate features and labels
X = data.drop("Class", axis=1)
y = data["Class"]

# 4. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Normalize data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 6. Use only normal transactions for training
X_train = X_train[y_train.values == 0]

# 7. Build Autoencoder
input_layer = Input(shape=(X_train.shape[1],))
encoded = Dense(14, activation='relu')(input_layer)
encoded = Dense(7, activation='relu')(encoded)
decoded = Dense(14, activation='relu')(encoded)
decoded = Dense(X_train.shape[1], activation='sigmoid')(decoded)
model = Model(input_layer, decoded)

# 8. Compile and train
model.compile(optimizer='adam', loss='mse')
model.fit(X_train, X_train, epochs=20, batch_size=64, 
          validation_split=0.1, verbose=1)

# 9. Calculate reconstruction error
reconstructed = model.predict(X_test)
error = np.mean((X_test - reconstructed) ** 2, axis=1)

# 10. Set a strict threshold (99.9th percentile)
train_pred = model.predict(X_train)
train_error = np.mean((X_train - train_pred) ** 2, axis=1)

threshold = np.percentile(train_error, 99.9)  # Changed from 95 to 99.9

# 11. Detect fraud
y_pred = (error > threshold).astype(int)

print("\n--- Detection Results ---")
print("Actual Fraud Cases in Test Set:", int(sum(y_test)))
print("Predicted Fraud Cases:", int(sum(y_pred)))


# 12. Plot results with a logarithmic scale
plt.figure(figsize=(10, 6))
plt.hist(error[y_test == 0], bins=50, alpha=0.6, label="Normal", color="blue")
plt.hist(error[y_test == 1], bins=50, alpha=0.6, label="Fraud", color="red")
plt.yscale('log')
plt.xlabel("Reconstruction Error")
plt.ylabel("Count (Log Scale)")
plt.title("Credit Card Fraud Detection (Log Scale with 99.9% Threshold)")
plt.legend()
plt.show()